In [2]:
!nvidia-smi

Sun Oct 18 03:25:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!pip install -q transformers datasets accelerate peft bitsandbytes scikit-learn

In [9]:
import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print(os.listdir(DATA_DIR))

['task1_validation.csv', 'task1_train.csv', 'task1_test.csv', 'task1_baseline_metrics.csv', 'task1_tfidf_logistic_baseline.joblib', 'teacher_pilot_100.csv', 'task1_label_audit_sample.csv', 'task1_labeling_rules.txt', 'task1_manual_review_candidates.csv', 'task1_train_validation_review.csv', 'task1_teacher_soft_targets_pilot.csv', 'task1_teacher_soft_targets_train.csv', 'task1_teacher_soft_targets_checkpoint.csv', 'native_student_model', 'native_student_training_history.csv', 'native_student_validation_audit.csv', 'distilled_student_model', 'distilled_student_training_history.csv', 'finetuned_teacher_7b']


In [10]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

for filename in [
    "task1_train.csv",
    "task1_validation.csv",
    "task1_test.csv"
]:
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)

    print(f"\n{filename}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df["label"].value_counts())


task1_train.csv
Shape: (4000, 2)
Columns: ['text', 'label']
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64

task1_validation.csv
Shape: (799, 2)
Columns: ['text', 'label']
label
DATABASE    400
GENERAL     399
Name: count, dtype: int64

task1_test.csv
Shape: (1200, 2)
Columns: ['text', 'label']
label
GENERAL     600
DATABASE    600
Name: count, dtype: int64


In [11]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_train.csv")
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_validation.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_test.csv")
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nSample rows:")
display(train_df.head())

Train: (4000, 2)
Validation: (799, 2)
Test: (1200, 2)

Training labels:
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64

Sample rows:


,text,label
0,how many coupless totalled 14.5,DATABASE
1,what are the atmospheric conditions in sparks,GENERAL
2,Can you tell me the highest Total votes that h...,DATABASE
3,what can you tell me about miami weather,GENERAL
4,i am needing to know the current time in the e...,GENERAL


In [13]:
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [14]:
import os

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "finetuned_teacher_7b"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL2ID = {
    "DATABASE": 0,
    "GENERAL": 1
}

ID2LABEL = {
    0: "DATABASE",
    1: "GENERAL"
}

print("Teacher:", MODEL_NAME)
print("Output:", OUTPUT_DIR)
print("Labels:", LABEL2ID)

Teacher: Qwen/Qwen2.5-7B-Instruct
Output: /content/drive/MyDrive/Task1_Research/finetuned_teacher_7b
Labels: {'DATABASE': 0, 'GENERAL': 1}


In [19]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "DATABASE", 1: "GENERAL"},
    label2id={"DATABASE": 0, "GENERAL": 1},
    quantization_config=bnb_config,
    device_map="auto"
)

teacher_model.config.pad_token_id = tokenizer.pad_token_id

print("Teacher model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Labels:", teacher_model.config.id2label)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-7B-Instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Teacher model loaded successfully!
GPU: Tesla T4
Labels: {0: 'DATABASE', 1: 'GENERAL'}


In [20]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

# Prepare quantized model for training
teacher_model.config.use_cache = False

teacher_model = prepare_model_for_kbit_training(
    teacher_model
)

# LoRA configuration
teacher_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    modules_to_save=["score"]
)

teacher_model = get_peft_model(
    teacher_model,
    teacher_lora_config
)

teacher_model.print_trainable_parameters()

trainable params: 20,192,256 || all params: 7,090,818,560 || trainable%: 0.2848


In [21]:
from datasets import Dataset

# Convert labels to class IDs
train_data = train_df.copy()
val_data = val_df.copy()

train_data["labels"] = train_data["label"].map(LABEL2ID)
val_data["labels"] = val_data["label"].map(LABEL2ID)

# Keep only required columns
train_data = Dataset.from_pandas(
    train_data[["text", "labels"]],
    preserve_index=False
)

val_data = Dataset.from_pandas(
    val_data[["text", "labels"]],
    preserve_index=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

train_data = train_data.map(
    tokenize_function,
    batched=True
)

val_data = val_data.map(
    tokenize_function,
    batched=True
)

print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))
print("Tokenized columns:", train_data.column_names)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/799 [00:00<?, ? examples/s]

Training rows: 4000
Validation rows: 799
Tokenized columns: ['text', 'labels', 'input_ids', 'attention_mask']


In [22]:
import inspect
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# Evaluate using validation data
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# T4-friendly training configuration
training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "fp16": True,
    "gradient_checkpointing": True,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_f1",
    "greater_is_better": True,
    "logging_steps": 25,
    "save_total_limit": 2,
    "report_to": "none",
    "optim": "paged_adamw_8bit",
    "seed": 42
}

# Support different Transformers versions
params = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in params:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

print("Training configuration ready!")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch size:", (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
))

Training configuration ready!
Epochs: 3
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8


In [23]:
trainer = Trainer(
    model=teacher_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

teacher_train_result = trainer.train()

print("\nTeacher fine-tuning completed!")

print("\nFinal validation metrics:")
print(trainer.evaluate())

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTeacher saved to:", OUTPUT_DIR)



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.045208,0.997497,0.997506,0.997500,0.997497
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.045208,0.997497,0.997506,0.997500,0.997497
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000



Teacher fine-tuning completed!

Final validation metrics:


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000000,0.000000,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.0, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}

Teacher saved to: /content/drive/MyDrive/Task1_Research/finetuned_teacher_7b


In [24]:
import torch
import torch.nn.functional as F
import pandas as pd
import os
import numpy as np

teacher_model.eval()

# Disable training-related behavior for inference
teacher_model.config.use_cache = False

device = next(teacher_model.parameters()).device
temperature = 2.0

all_probs = []

for start in range(0, len(train_df), 16):
    batch_df = train_df.iloc[start:start + 16]

    inputs = tokenizer(
        batch_df["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = teacher_model(**inputs)
        logits = outputs.logits.float()

        probs = F.softmax(
            logits / temperature,
            dim=-1
        )

    all_probs.extend(probs.cpu().numpy())

soft_targets_df = pd.DataFrame({
    "text": train_df["text"].tolist(),
    "label": train_df["label"].tolist(),
    "teacher_DATABASE": np.array(all_probs)[:, 0],
    "teacher_GENERAL": np.array(all_probs)[:, 1]
})

soft_targets_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_soft_targets_train.csv"
)

soft_targets_df.to_csv(
    soft_targets_path,
    index=False
)

print("Soft targets generated:", len(soft_targets_df))
print("Saved to:", soft_targets_path)
display(soft_targets_df.head())

Soft targets generated: 4000
Saved to: /content/drive/MyDrive/Task1_Research/finetuned_teacher_soft_targets_train.csv


,text,label,teacher_DATABASE,teacher_GENERAL
0,how many coupless totalled 14.5,DATABASE,1.000000e+00,2.103088e-18
1,what are the atmospheric conditions in sparks,GENERAL,1.228186e-10,1.000000e+00
2,Can you tell me the highest Total votes that h...,DATABASE,1.000000e+00,3.289783e-17
3,what can you tell me about miami weather,GENERAL,4.534574e-13,1.000000e+00
4,i am needing to know the current time in the e...,GENERAL,1.235219e-11,1.000000e+00


In [25]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

STUDENT_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

STUDENT_OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "finetuned_teacher_kd_student"
)
os.makedirs(STUDENT_OUTPUT_DIR, exist_ok=True)

# Load soft targets created by the fine-tuned teacher
soft_targets_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_soft_targets_train.csv"
)
soft_targets_df = pd.read_csv(soft_targets_path)

assert len(soft_targets_df) == len(train_df) == 4000
assert soft_targets_df[
    ["teacher_DATABASE", "teacher_GENERAL"]
].notna().all().all()

print("Soft targets loaded:", len(soft_targets_df))
print("Student output:", STUDENT_OUTPUT_DIR)

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

bnb_config_student = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    quantization_config=bnb_config_student,
    device_map="auto"
)

student_model.config.use_cache = False
student_model = prepare_model_for_kbit_training(student_model)

student_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

student_model = get_peft_model(
    student_model,
    student_lora_config
)

student_model.print_trainable_parameters()

Soft targets loaded: 4000
Student output: /content/drive/MyDrive/Task1_Research/finetuned_teacher_kd_student


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [26]:
database_id = student_tokenizer(
    "DATABASE",
    add_special_tokens=False
)["input_ids"]

general_id = student_tokenizer(
    "GENERAL",
    add_special_tokens=False
)["input_ids"]

print("DATABASE token IDs:", database_id)
print("GENERAL token IDs:", general_id)

assert len(database_id) == 1
assert len(general_id) == 1

DATABASE_ID = database_id[0]
GENERAL_ID = general_id[0]

DATABASE token IDs: [46499]
GENERAL token IDs: [94736]


In [27]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F
import numpy as np

class SoftTargetDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.targets = df[
            ["teacher_DATABASE", "teacher_GENERAL"]
        ].to_numpy(dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "soft_targets": torch.tensor(
                self.targets[idx],
                dtype=torch.float32
            )
        }

def collate_fn(batch):
    inputs = student_tokenizer.pad(
        [
            {
                "input_ids": x["input_ids"],
                "attention_mask": x["attention_mask"]
            }
            for x in batch
        ],
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt"
    )

    inputs["soft_targets"] = torch.stack(
        [x["soft_targets"] for x in batch]
    )

    return inputs

kd_train_dataset = SoftTargetDataset(
    soft_targets_df,
    student_tokenizer
)

kd_train_loader = DataLoader(
    kd_train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

print("KD training samples:", len(kd_train_dataset))
print("Batches per epoch:", len(kd_train_loader))

batch = next(iter(kd_train_loader))
print("Input shape:", batch["input_ids"].shape)
print("Soft-target shape:", batch["soft_targets"].shape)
print("Example soft target:", batch["soft_targets"][0].tolist())

KD training samples: 4000
Batches per epoch: 2000
Input shape: torch.Size([2, 24])
Soft-target shape: torch.Size([2, 2])
Example soft target: [1.0, 4.59356637925909e-18]


In [29]:
from torch.optim import AdamW
from tqdm.auto import tqdm
import gc

TEMPERATURE = 2.0
EPOCHS = 3
LEARNING_RATE = 2e-4
GRAD_ACCUM_STEPS = 8

optimizer = AdamW(
    [p for p in student_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

student_model.config.use_cache = False
student_model.train()

student_device = next(student_model.parameters()).device

history = []

for epoch in range(EPOCHS):
    student_model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(kd_train_loader),
        total=len(kd_train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in progress:
        input_ids = batch["input_ids"].to(student_device)
        attention_mask = batch["attention_mask"].to(student_device)
        teacher_probs = batch["soft_targets"].to(student_device)

        outputs = student_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Last non-padding token's next-token logits
        last_positions = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(
            input_ids.size(0),
            device=student_device
        )

        last_logits = outputs.logits[
            batch_indices,
            last_positions
        ].float()

        student_class_logits = torch.stack(
            [
                last_logits[:, DATABASE_ID],
                last_logits[:, GENERAL_ID]
            ],
            dim=1
        )

        # Temperature-scaled student distribution
        student_log_probs = F.log_softmax(
            student_class_logits / TEMPERATURE,
            dim=-1
        )

        # KL divergence: teacher distribution -> student distribution
        loss = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="batchmean"
        ) * (TEMPERATURE ** 2)

        (loss / GRAD_ACCUM_STEPS).backward()
        total_loss += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                student_model.parameters(),
                max_norm=1.0
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{loss.item():.4f}")

    # Handle any remaining accumulated gradients
    if (step + 1) % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(
            student_model.parameters(),
            max_norm=1.0
        )
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    avg_loss = total_loss / len(kd_train_loader)
    history.append({
        "epoch": epoch + 1,
        "train_kl_loss": avg_loss
    })

    print(f"Epoch {epoch + 1} — average KL loss: {avg_loss:.6f}")

    student_model.save_pretrained(STUDENT_OUTPUT_DIR)
    student_tokenizer.save_pretrained(STUDENT_OUTPUT_DIR)

    pd.DataFrame(history).to_csv(
        os.path.join(STUDENT_OUTPUT_DIR, "training_history.csv"),
        index=False
    )

print("KD student training completed.")

Epoch 1/3:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch 1 — average KL loss: 0.033749


Epoch 2/3:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch 2 — average KL loss: 0.014412


Epoch 3/3:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch 3 — average KL loss: 0.029193
KD student training completed.


In [30]:
import numpy as np
import torch
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

student_model.eval()
student_tokenizer.padding_side = "right"

device = next(student_model.parameters()).device
all_predictions = []
all_true_labels = []
all_confidences = []

label2id = {"DATABASE": 0, "GENERAL": 1}
id2label = {0: "DATABASE", 1: "GENERAL"}

batch_size = 8

for start in range(0, len(val_df), batch_size):
    batch_df = val_df.iloc[start:start + batch_size]

    inputs = student_tokenizer(
        batch_df["text"].astype(str).tolist(),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = student_model(**inputs)

        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        batch_indices = torch.arange(
            inputs["input_ids"].size(0),
            device=device
        )

        last_logits = outputs.logits[
            batch_indices, last_positions
        ].float()

        class_logits = torch.stack(
            [
                last_logits[:, DATABASE_ID],
                last_logits[:, GENERAL_ID]
            ],
            dim=1
        )

        probs = torch.softmax(class_logits, dim=1)
        preds = probs.argmax(dim=1)

    all_predictions.extend(preds.cpu().tolist())
    all_confidences.extend(probs.max(dim=1).values.cpu().tolist())
    all_true_labels.extend(
        batch_df["label"].map(label2id).tolist()
    )

accuracy = accuracy_score(all_true_labels, all_predictions)

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    all_true_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

print(f"Validation records: {len(all_true_labels)}")
print(f"Accuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Macro F1:  {macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    all_true_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(confusion_matrix(
    all_true_labels,
    all_predictions,
    labels=[0, 1]
))

Validation records: 799
Accuracy:  0.9975 (99.75%)
Precision: 0.9975
Recall:    0.9975
Macro F1:  0.9975

Classification report:
              precision    recall  f1-score   support

    DATABASE     1.0000    0.9950    0.9975       400
     GENERAL     0.9950    1.0000    0.9975       399

    accuracy                         0.9975       799
   macro avg     0.9975    0.9975    0.9975       799
weighted avg     0.9975    0.9975    0.9975       799

Confusion matrix (actual rows, predicted columns):
[[398   2]
 [  0 399]]


In [31]:
import numpy as np
import torch
import pandas as pd
import os

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

student_model.eval()

label2id = {"DATABASE": 0, "GENERAL": 1}
id2label = {0: "DATABASE", 1: "GENERAL"}

test_batch_size = 8
test_predictions = []
test_confidences = []
test_true_labels = []

device = next(student_model.parameters()).device

for start in range(0, len(test_df), test_batch_size):
    batch_df = test_df.iloc[start:start + test_batch_size]

    inputs = student_tokenizer(
        batch_df["text"].astype(str).tolist(),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = student_model(**inputs)

        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        batch_indices = torch.arange(
            inputs["input_ids"].size(0), device=device
        )

        last_logits = outputs.logits[
            batch_indices, last_positions
        ].float()

        class_logits = torch.stack(
            [last_logits[:, DATABASE_ID],
             last_logits[:, GENERAL_ID]],
            dim=1
        )

        probs = torch.softmax(class_logits, dim=1)
        preds = probs.argmax(dim=1)

    test_predictions.extend(preds.cpu().tolist())
    test_confidences.extend(probs.max(dim=1).values.cpu().tolist())
    test_true_labels.extend(
        batch_df["label"].map(label2id).tolist()
    )

accuracy = accuracy_score(test_true_labels, test_predictions)

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    test_true_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print(f"Test records: {len(test_true_labels)}")
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    test_true_labels,
    test_predictions,
    labels=[0, 1],
    target_names=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    test_true_labels,
    test_predictions,
    labels=[0, 1]
))

# Save predictions and metrics
test_predictions_df = test_df.copy()
test_predictions_df["predicted_label"] = [
    id2label[p] for p in test_predictions
]
test_predictions_df["confidence"] = test_confidences

test_predictions_df.to_csv(
    os.path.join(DATA_DIR, "finetuned_teacher_kd_student_test_predictions.csv"),
    index=False
)

test_metrics = pd.DataFrame([{
    "model": "Fine-tuned Teacher KD SLM",
    "accuracy": accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": macro_f1
}])

test_metrics.to_csv(
    os.path.join(DATA_DIR, "finetuned_teacher_kd_student_test_metrics.csv"),
    index=False
)

print("\nTest results saved to Google Drive.")

Test records: 1200
Accuracy: 0.9992 (99.92%)
Macro Precision: 0.9992
Macro Recall: 0.9992
Macro F1: 0.9992

Classification Report:
              precision    recall  f1-score   support

    DATABASE     1.0000    0.9983    0.9992       600
     GENERAL     0.9983    1.0000    0.9992       600

    accuracy                         0.9992      1200
   macro avg     0.9992    0.9992    0.9992      1200
weighted avg     0.9992    0.9992    0.9992      1200


Confusion Matrix:
[[599   1]
 [  0 600]]

Test results saved to Google Drive.


In [32]:
import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print("Files and folders:")
for name in os.listdir(DATA_DIR):
    print(name)

Files and folders:
task1_validation.csv
task1_train.csv
task1_test.csv
task1_baseline_metrics.csv
task1_tfidf_logistic_baseline.joblib
teacher_pilot_100.csv
task1_label_audit_sample.csv
task1_labeling_rules.txt
task1_manual_review_candidates.csv
task1_train_validation_review.csv
task1_teacher_soft_targets_pilot.csv
task1_teacher_soft_targets_train.csv
task1_teacher_soft_targets_checkpoint.csv
native_student_model
native_student_training_history.csv
native_student_validation_audit.csv
distilled_student_model
distilled_student_training_history.csv
finetuned_teacher_7b
finetuned_teacher_soft_targets_train.csv
finetuned_teacher_kd_student
finetuned_teacher_kd_student_test_predictions.csv
finetuned_teacher_kd_student_test_metrics.csv


In [33]:
import os
import gc
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
NATIVE_PATH = os.path.join(DATA_DIR, "native_student_model")

# Load the saved Native SLM
native_tokenizer = AutoTokenizer.from_pretrained(NATIVE_PATH)

if native_tokenizer.pad_token is None:
    native_tokenizer.pad_token = native_tokenizer.eos_token

native_tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

native_model = PeftModel.from_pretrained(
    base_model,
    NATIVE_PATH
)

native_model.eval()

DATABASE_ID = native_tokenizer(
    "DATABASE", add_special_tokens=False
)["input_ids"][0]

GENERAL_ID = native_tokenizer(
    "GENERAL", add_special_tokens=False
)["input_ids"][0]

assert len(native_tokenizer(
    "DATABASE", add_special_tokens=False
)["input_ids"]) == 1

assert len(native_tokenizer(
    "GENERAL", add_special_tokens=False
)["input_ids"]) == 1


# Use the same instruction prompt as Native SLM training
def make_prompt(query):
    messages = [
        {
            "role": "user",
            "content": (
                "You are a semantic query classifier for a database system.\n\n"
                "Classify the user's query into exactly one of these labels:\n"
                "DATABASE\nGENERAL\n\n"
                "DATABASE:\n"
                "The query requests information that would normally be retrieved "
                "from structured database records, tables, or fields. It may use "
                "natural language and does not need to mention SQL or a database "
                "explicitly. Examples include asking for a person's recorded "
                "country, a team's score, a town's latitude, a ranking, a count, "
                "or a record matching conditions.\n\n"
                "GENERAL:\n"
                "The query is ordinary conversation, entertainment, translation, "
                "general explanation, or advice, and does not request a specific "
                "structured record. Examples include asking for a joke, "
                "translating a word, or requesting music.\n\n"
                "Do not classify a query as GENERAL merely because it is written "
                "as a natural-language question.\n"
                "Return only DATABASE or GENERAL.\n\n"
                f"Query: {query}\nLabel:"
            )
        }
    ]

    return native_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


device = next(native_model.parameters()).device
predictions = []
true_labels = []

test_batch_size = 8

for start in range(0, len(test_df), test_batch_size):
    batch_df = test_df.iloc[start:start + test_batch_size]

    prompts = [
        make_prompt(q)
        for q in batch_df["text"].astype(str).tolist()
    ]

    inputs = native_tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = native_model(**inputs)

        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        batch_indices = torch.arange(
            inputs["input_ids"].size(0),
            device=device
        )

        last_logits = outputs.logits[
            batch_indices, last_positions
        ].float()

        class_logits = torch.stack(
            [
                last_logits[:, DATABASE_ID],
                last_logits[:, GENERAL_ID]
            ],
            dim=1
        )

        batch_predictions = class_logits.argmax(dim=1)

    predictions.extend(batch_predictions.cpu().tolist())
    true_labels.extend(
        batch_df["label"].map(
            {"DATABASE": 0, "GENERAL": 1}
        ).tolist()
    )

accuracy = accuracy_score(true_labels, predictions)

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="macro",
    zero_division=0
)

print("Native SLM — Test Results")
print("Test records:", len(true_labels))
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    true_labels,
    predictions,
    labels=[0, 1],
    target_names=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(true_labels, predictions, labels=[0, 1]))

native_results = pd.DataFrame([{
    "model": "Native SLM",
    "accuracy": accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": macro_f1
}])

native_results.to_csv(
    os.path.join(DATA_DIR, "native_student_test_metrics.csv"),
    index=False
)

print("\nMetrics saved to Google Drive.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Native SLM — Test Results
Test records: 1200
Accuracy: 0.9958 (99.58%)
Macro Precision: 0.9958
Macro Recall: 0.9958
Macro F1: 0.9958

Classification Report:
              precision    recall  f1-score   support

    DATABASE     0.9934    0.9983    0.9958       600
     GENERAL     0.9983    0.9933    0.9958       600

    accuracy                         0.9958      1200
   macro avg     0.9958    0.9958    0.9958      1200
weighted avg     0.9958    0.9958    0.9958      1200

Confusion Matrix:
[[599   1]
 [  4 596]]

Metrics saved to Google Drive.


In [34]:
import os
import gc
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
KD_PATH = os.path.join(DATA_DIR, "distilled_student_model")

# Free memory from previously loaded models if present
for name in ["native_model", "base_model", "student_model", "kd_model", "kd_base"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load test data if it is not already in memory
if "test_df" not in globals():
    test_df = pd.read_csv(
        os.path.join(DATA_DIR, "task1_test.csv")
    )

# Load the original KD model and its tokenizer
kd_tokenizer = AutoTokenizer.from_pretrained(KD_PATH)

if kd_tokenizer.pad_token is None:
    kd_tokenizer.pad_token = kd_tokenizer.eos_token

kd_tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

kd_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

kd_model = PeftModel.from_pretrained(kd_base, KD_PATH)
kd_model.eval()

DATABASE_ID = kd_tokenizer(
    "DATABASE", add_special_tokens=False
)["input_ids"][0]

GENERAL_ID = kd_tokenizer(
    "GENERAL", add_special_tokens=False
)["input_ids"][0]

assert len(kd_tokenizer(
    "DATABASE", add_special_tokens=False
)["input_ids"]) == 1
assert len(kd_tokenizer(
    "GENERAL", add_special_tokens=False
)["input_ids"]) == 1

device = next(kd_model.parameters()).device
y_true = test_df["label"].map({
    "DATABASE": 0,
    "GENERAL": 1
}).tolist()

y_pred = []

# IMPORTANT: raw query text, matching original KD training
for start in range(0, len(test_df), 8):
    batch_df = test_df.iloc[start:start + 8]

    inputs = kd_tokenizer(
        batch_df["text"].astype(str).tolist(),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = kd_model(**inputs)

        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        batch_indices = torch.arange(
            inputs["input_ids"].size(0),
            device=device
        )

        last_logits = outputs.logits[
            batch_indices, last_positions
        ].float()

        class_logits = torch.stack([
            last_logits[:, DATABASE_ID],
            last_logits[:, GENERAL_ID]
        ], dim=1)

        preds = class_logits.argmax(dim=1)

    y_pred.extend(preds.cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

print("Original KD SLM — Test Results")
print("Test records:", len(y_true))
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

pd.DataFrame([{
    "model": "Original KD SLM",
    "accuracy": accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": macro_f1
}]).to_csv(
    os.path.join(DATA_DIR, "original_kd_student_test_metrics.csv"),
    index=False
)

print("\nMetrics saved to Google Drive.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Original KD SLM — Test Results
Test records: 1200
Accuracy: 0.5925 (59.25%)
Macro Precision: 0.6518
Macro Recall: 0.5925
Macro F1: 0.5484

Classification Report:
              precision    recall  f1-score   support

    DATABASE     0.5569    0.9050    0.6895       600
     GENERAL     0.7467    0.2800    0.4073       600

    accuracy                         0.5925      1200
   macro avg     0.6518    0.5925    0.5484      1200
weighted avg     0.6518    0.5925    0.5484      1200

Confusion Matrix:
[[543  57]
 [432 168]]

Metrics saved to Google Drive.


In [35]:
import os
import re
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

train_df = pd.read_csv(os.path.join(DATA_DIR, "task1_train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "task1_validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "task1_test.csv"))

# Normalize text to detect exact duplicates despite case/spacing differences
def normalize_query(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

train_queries = train_df["text"].map(normalize_query)
val_queries = val_df["text"].map(normalize_query)
test_queries = test_df["text"].map(normalize_query)

# Check duplicates within each split
print("Duplicate queries within each split:")
print("Train:", train_queries.duplicated().sum())
print("Validation:", val_queries.duplicated().sum())
print("Test:", test_queries.duplicated().sum())

# Check overlap between splits
train_set = set(train_queries)
val_set = set(val_queries)
test_set = set(test_queries)

train_val = train_set & val_set
train_test = train_set & test_set
val_test = val_set & test_set

print("\nOverlap between splits:")
print("Train ∩ Validation:", len(train_val))
print("Train ∩ Test:", len(train_test))
print("Validation ∩ Test:", len(val_test))

# Save any overlapping queries for inspection
overlap_rows = []

for split_a, split_b, overlap in [
    ("Train", "Validation", train_val),
    ("Train", "Test", train_test),
    ("Validation", "Test", val_test),
]:
    for query in sorted(overlap):
        overlap_rows.append({
            "split_a": split_a,
            "split_b": split_b,
            "normalized_query": query
        })

overlap_path = os.path.join(
    DATA_DIR, "task1_data_leakage_audit.csv"
)

pd.DataFrame(
    overlap_rows,
    columns=["split_a", "split_b", "normalized_query"]
).to_csv(overlap_path, index=False)

print("\nAudit saved to:", overlap_path)

Duplicate queries within each split:
Train: 0
Validation: 0
Test: 0

Overlap between splits:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Audit saved to: /content/drive/MyDrive/Task1_Research/task1_data_leakage_audit.csv


In [36]:
import os
import json

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

for folder in [
    "native_student_model",
    "distilled_student_model",
    "finetuned_teacher_kd_student"
]:
    path = os.path.join(DATA_DIR, folder)

    print(f"\n--- {folder} ---")

    config_path = os.path.join(path, "adapter_config.json")

    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)

        print("Base model:", config.get("base_model_name_or_path"))
        print("Task type:", config.get("task_type"))
        print("PEFT type:", config.get("peft_type"))
    else:
        print("adapter_config.json not found")


--- native_student_model ---
Base model: Qwen/Qwen2.5-0.5B-Instruct
Task type: CAUSAL_LM
PEFT type: LORA

--- distilled_student_model ---
Base model: Qwen/Qwen2.5-0.5B-Instruct
Task type: CAUSAL_LM
PEFT type: LORA

--- finetuned_teacher_kd_student ---
Base model: Qwen/Qwen2.5-0.5B-Instruct
Task type: CAUSAL_LM
PEFT type: LORA
